In [1]:
using PowerDynamics
using PowerDynamics.Library
using ModelingToolkit
using ModelingToolkit: t_nounits as t, D_nounits as Dt
using OrdinaryDiffEqRosenbrock
using OrdinaryDiffEqNonlinearSolve
using CairoMakie
using NetworkDynamics
using DataFrames
using CSV


In [2]:
BASE_MVA = 100
BASE_FREQ = 60.0

path_to_csv_files = "dynamic-equivalents-data-export\\"

"dynamic-equivalents-data-export\\"

# Read System Data from CSV

In [3]:
using CSV

function parse_python_complex(s::String)
    s = strip(s, ['(', ')', ' '])
    m = match(r"([+-]?[\d.e+-]+)([+-][\d.e+-]+)j", s)
    if m !== nothing
        return complex(parse(Float64, m[1]), parse(Float64, m[2]))
    else
        # Handle cases like "0j"
        s = replace(s, "j" => "im")
        return parse(ComplexF64, s)
    end
end


admittance_matrix_df = CSV.read("$(path_to_csv_files)reduced-admittance-matrix.csv", DataFrame, header=1, drop=[1])
admittance_matrix_df = mapcols(col -> parse_python_complex.(col), admittance_matrix_df)
admittance_matrix_df

Row,Cluster 1,Cluster 2,Cluster 3,Cluster 5,Cluster 7
,Complex…,Complex…,Complex…,Complex…,Complex…
1,6.96316-33.4433im,-2.54431+10.9053im,-3.67427+17.9776im,-0.380078+1.89697im,-0.364499+2.6634im
2,-2.54431+10.9053im,3.24216-53.321im,0.679028+1.94108im,-1.51324+40.1648im,0.136374+0.309898im
3,-3.67427+17.9776im,0.679028+1.94108im,4.28104-52.3382im,-0.971144+12.4244im,-0.314657+19.9951im
4,-0.380078+1.89697im,-1.51324+40.1648im,-0.971144+12.4244im,2.8798-59.8509im,-0.0153353+5.36475im
5,-0.364499+2.6634im,0.136374+0.309898im,-0.314657+19.9951im,-0.0153353+5.36475im,0.558117-28.3331im


In [4]:
admittance_matrix = Matrix(admittance_matrix_df)
admittance_matrix

5×5 Matrix{ComplexF64}:
   6.96316-33.4433im  -2.54431+10.9053im   …   -0.364499+2.6634im
  -2.54431+10.9053im   3.24216-53.321im         0.136374+0.309898im
  -3.67427+17.9776im  0.679028+1.94108im       -0.314657+19.9951im
 -0.380078+1.89697im  -1.51324+40.1648im      -0.0153353+5.36475im
 -0.364499+2.6634im   0.136374+0.309898im       0.558117-28.3331im

In [5]:
power_exchange_df = CSV.read("$(path_to_csv_files)power-exchange.csv", DataFrame, header=1, drop=[1])
power_exchange_df = coalesce.(power_exchange_df, "0+0j")
power_exchange_df = mapcols(col -> parse_python_complex.(col), power_exchange_df)
power_exchange_matrix = Matrix(power_exchange_df)
power_exchange_df

Row,Cluster 1,Cluster 2,Cluster 3,Cluster 5,Cluster 7
,Complex…,Complex…,Complex…,Complex…,Complex…
1,0.0+0.0im,0.0+0.0im,314.195-55.2207im,0.0+0.0im,0.0+0.0im
2,0.0+0.0im,0.0+0.0im,-104.0-161.719im,0.0+0.0im,0.0+0.0im
3,-314.195+55.2207im,104.0+161.719im,0.0+0.0im,-511.611-193.652im,-250.0-146.158im
4,0.0+0.0im,0.0+0.0im,511.611+193.652im,0.0+0.0im,0.0+0.0im
5,0.0+0.0im,0.0+0.0im,250.0+146.158im,0.0+0.0im,0.0+0.0im


In [6]:
load_power_df = CSV.read("$(path_to_csv_files)total-power-loads.csv", DataFrame, header=["subsystem", "total_load"], skipto=2)
load_power_df.total_load = parse_python_complex.(load_power_df.total_load)
load_power_vector = Vector(load_power_df.total_load)

5-element Vector{ComplexF64}:
             224.0 + 47.20000076293945im
            1104.0 + 250.00000000000003im
 4759.900009155273 + 1107.1000008583069im
 9.199999809265137 + 4.599999904632568im
               0.0 + 0.0im

In [7]:
total_power_exchange_subsystems = vec(sum(power_exchange_matrix, dims=1))
total_generation_subsystems = load_power_vector - total_power_exchange_subsystems
total_generation_subsystems

5-element Vector{ComplexF64}:
 538.1952856914852 - 8.020670402039315im
 999.9999999941572 + 88.28141463402508im
 3788.093651146509 + 984.2292946405104im
 520.8110721392796 + 198.25178449871325im
 249.9999999931074 + 146.1581781546695im

# Create Lines 

In [8]:
pi_line_template = MTKLine(Library.PiLine(; name=:piline))

num_buses = size(admittance_matrix, 1)

# Maximum possible number of branches (upper triangle)
max_lines = (num_buses * (num_buses - 1)) ÷ 2

lines = Vector{EdgeModel}(undef, max_lines)

k = 1
@inbounds for j in 2:num_buses
    for i in 1:j-1
        Y = admittance_matrix[i, j]
        if Y != 0
            Z = -inv(Y)  # convert admittance → impedance
            lines[k] = compile_line(
                pi_line_template;
                src=i,
                dst=j,
                piline₊R=real(Z),
                piline₊X=imag(Z),
                name=Symbol("line_$(i)_$(j)")
            )
            k += 1
        end
    end
end

# Trim unused entries (if zeros existed)
resize!(lines, k - 1)

10-element Vector{EdgeModel}:
 EdgeModel :line_1_2 @ Edge 1=>2
 EdgeModel :line_1_3 @ Edge 1=>3
 EdgeModel :line_2_3 @ Edge 2=>3
 EdgeModel :line_1_4 @ Edge 1=>4
 EdgeModel :line_2_4 @ Edge 2=>4
 EdgeModel :line_3_4 @ Edge 3=>4
 EdgeModel :line_1_5 @ Edge 1=>5
 EdgeModel :line_2_5 @ Edge 2=>5
 EdgeModel :line_3_5 @ Edge 3=>5
 EdgeModel :line_4_5 @ Edge 4=>5

# Create Buses

In [9]:
@named synchronous_machine_swing = Library.Swing(V=1)
@named load_pq = PQLoad()

@named machine_load_bus_template = compile_bus(
    MTKBus(synchronous_machine_swing, load_pq);
)

strip_defaults!(machine_load_bus_template)
# set_default!(machine_load_bus_template, r"S_b$", BASE_MVA)
# set_default!(machine_load_bus_template, r"ω_b$", 2π*BASE_FREQ)



VertexModel :machine_load_bus_template NoFeedForward()
 ├─ 2 inputs:  [busbar₊i_r, busbar₊i_i]
 ├─ 2 states:  [synchronous_machine_swing₊ω≈0, synchronous_machine_swing₊θ≈0]
 ├─ 2 outputs: [busbar₊u_r, busbar₊u_i]
 └─ 7 params:  [synchronous_machine_swing₊ω_ref, synchronous_machine_swing₊Pm≈1, synchronous_machine_swing₊M, synchronous_machine_swing₊D, synchronous_machine_swing₊V≈1, load_pq₊Pset, load_pq₊Qset]

In [10]:
total_power_exchange_subsystems

5-element Vector{ComplexF64}:
 -314.19528569148525 + 55.22067116497877im
  104.00000000584281 + 161.71858536597495im
   971.8063580087644 + 122.87070621779648im
  -511.6110723300145 - 193.65178459408068im
  -249.9999999931074 - 146.1581781546695im

In [11]:
total_generation_subsystems_pu = total_generation_subsystems ./ BASE_MVA
load_power_vector_pu = load_power_vector ./ BASE_MVA

5-element Vector{ComplexF64}:
                2.24 + 0.47200000762939454im
               11.04 + 2.5000000000000004im
   47.59900009155273 + 11.071000008583068im
 0.09199999809265137 + 0.045999999046325686im
                 0.0 + 0.0im

Compile buses and add power flow model. For the power flow models, the first bus is the slack and the rest are PQ controlled. Additional constrains ares added to the loads for their power output. This should be enough information to solve the initialization of buses and to share as the power between the Swing and load model.

In [17]:
num_buses = size(admittance_matrix, 1)

buses = Vector{VertexModel}(undef, num_buses)

for bus in 1:num_buses
    buses[bus] = compile_bus(machine_load_bus_template; vidx=bus, name=Symbol("bus_$(bus)"))
    if bus == 1
        pf_model = pfSlack(V=1, δ=0)
    else
        p_subsystem = real(total_power_exchange_subsystems[bus])
        q_subsystem = imag(total_power_exchange_subsystems[bus])
        pf_model = pfPQ(P=p_subsystem, Q=q_subsystem)
    end
    set_pfmodel!(buses[bus], pf_model)
    p_load = real(load_power_vector_pu[bus])
    q_load = imag(load_power_vector_pu[bus])
    constraint = @initconstraint begin
        :load_pq₊Pset - p_load
        :load_pq₊Qset - q_load
    end
    set_initconstraint!(buses[bus], constraint)
end
buses

5-element Vector{VertexModel}:
 VertexModel :bus_1 @ Vertex 1
 VertexModel :bus_2 @ Vertex 2
 VertexModel :bus_3 @ Vertex 3
 VertexModel :bus_4 @ Vertex 4
 VertexModel :bus_5 @ Vertex 5

# Create Network

In [18]:
nw = Network(buses, lines)

┌ Warning: Order of edge models was changed to match the natural ordering of edges in graph (as in `edges(g)`)! Concretely, this means that `nw[EIndex(1)]` references the first edge from `edges(g)` **and not necessarily the first edge model in the provided list**. Disable warning with kw `warn_order=false`.
└ @ NetworkDynamics C:\Users\seberlein\.julia\packages\NetworkDynamics\yJyFn\src\construction.jl:287


Network with 10 states and 125 parameters
 ├─ 5 vertices (1 unique type)
 └─ 10 edges (1 unique type)
Edge-Aggregation using SequentialAggregator(+)

# Solve Power Flow

In [21]:
pfs = solve_powerflow(nw)

LoadError: MethodError: to_indices(::Vector{NetworkDynamics.SymbolicIndex}, ::Tuple{Vector{Union{}}}) is ambiguous.

Candidates:
  to_indices([90mA[39m, [90mI[39m::[1mTuple[22m[0m{AbstractVector{<:BlockArrays.Block{1}}, Vararg{Any}})
[90m    @[39m [90mBlockArrays[39m [90mC:\Users\seberlein\.julia\packages\BlockArrays\McfNk\src\[39m[90m[4mviews.jl:69[24m[39m
  to_indices([90mA[39m, [90mI[39m::[1mTuple[22m[0m{AbstractVector{<:BlockArrays.BlockRange{1, R} where R<:Tuple{AbstractUnitRange{<:Integer}}}, Vararg{Any}})
[90m    @[39m [90mBlockArrays[39m [90mC:\Users\seberlein\.julia\packages\BlockArrays\McfNk\src\[39m[90m[4mviews.jl:70[24m[39m
  to_indices([90mA[39m, [90mI[39m::[1mTuple[22m[0m{AbstractVector{<:BlockArrays.BlockIndexRange{1, R, I} where {R<:Tuple{AbstractUnitRange{<:Integer}}, I<:Tuple{Integer}}}, Vararg{Any}})
[90m    @[39m [90mBlockArrays[39m [90mC:\Users\seberlein\.julia\packages\BlockArrays\McfNk\src\[39m[90m[4mviews.jl:73[2

In [22]:
s0 = initialize_from_pf!(nw)

┌ Warning: The `alias_u0` keyword argument is deprecated. Please use a NonlinearAliasSpecifier, e.g. `alias = NonlinearAliasSpecifier(alias_u0 = true)`.
└ @ NonlinearSolveBase C:\Users\seberlein\.julia\packages\NonlinearSolveBase\2E600\src\solve.jl:57


LoadError: NetworkInitError: Could not find fixpoint, solver returned MaxIters (alg=SteadyStateDiffEq.SSRootfind{Nothing}(nothing))! For debugging, it is advised to manually construct the steady state problem and try different solvers/arguments:

prob = SteadyStateProblem(nw, uflat(nwstate), pflat(nwpara))
sol = solve(prob, alg; kwargs...)
x0 = NWState(nw, sol.u)

For detail see https://docs.sciml.ai/NonlinearSolve/stable/native/steadystatediffeq/
